# 과제 1 - 회귀분석
### delivery_eta_train.csv를 이용하여 delivery_eta_test의 배달시간을 예측하는 회귀모델 작성 --> 예측 수행

#### delivery_dataspec(데이터명세서) 참고
- 타깃(예측할 값)
- Delivery_Time_min : 주문 후 고객에게 도착하기까지 총 소요 시간(분)
- 특징(입력 변수) 예시
- Distance_km : 거리(km)
- Prep_Time_min : 음식 준비시간(분)
- Riders_Available : 주변 가용 라이더 수(명)
- Order_Queue : 매장 주문 적체(건)
- Weather : 날씨(맑음/비/눈/안개)
- Rain_mm : 강수량(mm)
- Temp_C : 기온(°C)
- Wind_mps : 풍속(m/s)
- Rush_Hour : 러시아워 여부(0/1)
- DayOfWeek : 요일(0=월 … 6=일)
- Time_Slot : 시간대(Morning/Lunch/Afternoon/Dinner/Night)
- Vehicle : 배달수단(Bike/Scooter/Car)
- Road_Type : 도로환경(Downtown/Suburban/Highway)
- Promo : 프로모션 여부(0/1)
- Tip_Level : 팁 수준(0~3)

- 준비: delivery_eta_train.csv, delivery_eta_test.csv, delivery_dataspec 로드하고 타깃을 Delivery_Time_min, 식별자를 Order_ID로 확정.
- 데이터 점검: shape, dtypes, 결측치 비율, 중복 여부를 확인하고 로그로 정리.
- EDA: 타깃 분포(히스토그램), 수치형 상관관계, 범주형별 평균 ETA를 확인해 중요한 변수 후보를 기록.
- 피처 정의: X/y 분리, Order_ID는 X에서 제외, 수치형/범주형 컬럼 리스트를 고정.
- 전처리 설계: ColumnTransformer 구성(수치형 StandardScaler, 범주형 OneHotEncoder(handle_unknown='ignore')).
- 베이스라인 모델: LinearRegression, RandomForestRegressor, XGBRegressor를 각각 Pipeline으로 묶어 동일 조건에서 비교.
- 검증: KFold(n_splits=5, shuffle=True, random_state=42)로 교차검증, 1순위 RMSE, 보조 MAE, R2 기록.
- 모델 선택: 평균 RMSE가 가장 낮은 모델 1개를 선택하고 결과표(모델별 지표) 작성.
- 소규모 튜닝: 선택 모델 핵심 파라미터만 RandomizedSearchCV로 탐색해 성능 개선 여부 확인.
- 최종 학습/예측: train 전체로 재학습 후 test 예측 생성, Order_ID + ETA_pred 형태 결과 파일 저장.
- 사후 평가: test의 실제 Delivery_Time_min은 최종 점검 용도로만 사용해 RMSE/MAE/R2 계산.

In [1]:
# 커널 동작 확인
print("hello")

hello


In [2]:
# ./data/delivery_eta_train.csv csv 불러오기
import pandas as pd
train = pd.read_csv("./data/delivery_eta_train.csv")
train.head()

,Order_ID,Distance_km,Prep_Time_min,Riders_Available,Order_Queue,Weather,Rain_mm,Temp_C,Wind_mps,Rush_Hour,DayOfWeek,Time_Slot,Vehicle,Road_Type,Promo,Tip_Level,Delivery_Time_min
0,11502,6.1,15,9,3,Clear,0.0,14.9,1.2,1,1,Dinner,Scooter,Suburban,0,2,59.7
1,12587,7.5,14,15,6,Clear,0.0,14.3,3.0,0,0,Lunch,Scooter,Downtown,1,3,62.9
2,12654,17.6,24,14,4,Rain,4.8,27.0,3.2,0,0,Dinner,Bike,Downtown,0,2,134.7
3,11056,6.8,15,11,5,Clear,0.0,9.2,5.4,0,3,Night,Scooter,Highway,0,0,60.1
4,10706,19.6,16,11,2,Clear,0.0,11.8,2.6,1,2,Dinner,Bike,Downtown,0,1,149.2


In [3]:
# 데이터 정보 확인
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Order_ID           4000 non-null   int64  
 1   Distance_km        4000 non-null   float64
 2   Prep_Time_min      4000 non-null   int64  
 3   Riders_Available   4000 non-null   int64  
 4   Order_Queue        4000 non-null   int64  
 5   Weather            4000 non-null   str    
 6   Rain_mm            4000 non-null   float64
 7   Temp_C             4000 non-null   float64
 8   Wind_mps           4000 non-null   float64
 9   Rush_Hour          4000 non-null   int64  
 10  DayOfWeek          4000 non-null   int64  
 11  Time_Slot          4000 non-null   str    
 12  Vehicle            4000 non-null   str    
 13  Road_Type          4000 non-null   str    
 14  Promo              4000 non-null   int64  
 15  Tip_Level          4000 non-null   int64  
 16  Delivery_Time_min  4000 non-null   

In [4]:
# ./data/delivery_eta_test.csv csv 불러오기
test = pd.read_csv("./data/delivery_eta_test.csv")
test.head()

,Order_ID,Distance_km,Prep_Time_min,Riders_Available,Order_Queue,Weather,Rain_mm,Temp_C,Wind_mps,Rush_Hour,DayOfWeek,Time_Slot,Vehicle,Road_Type,Promo,Tip_Level,Delivery_Time_min
0,11702,16.4,14,7,2,Rain,2.8,34.3,2.2,0,2,Dinner,Car,Downtown,1,1,100.5
1,13294,5.3,16,14,4,Clear,0.0,7.0,2.6,1,2,Lunch,Bike,Downtown,0,3,66.3
2,12162,5.4,13,8,6,Rain,1.7,22.6,3.6,1,4,Dinner,Bike,Downtown,0,1,76.3
3,11137,20.0,17,3,1,Rain,4.8,28.0,3.0,1,2,Lunch,Scooter,Suburban,0,1,146.4
4,10988,7.4,16,11,0,Snow,8.1,21.4,2.8,0,0,Dinner,Scooter,Downtown,0,2,79.1


In [5]:
# 데이터 정보 확인
test.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Order_ID           1000 non-null   int64  
 1   Distance_km        1000 non-null   float64
 2   Prep_Time_min      1000 non-null   int64  
 3   Riders_Available   1000 non-null   int64  
 4   Order_Queue        1000 non-null   int64  
 5   Weather            1000 non-null   str    
 6   Rain_mm            1000 non-null   float64
 7   Temp_C             1000 non-null   float64
 8   Wind_mps           1000 non-null   float64
 9   Rush_Hour          1000 non-null   int64  
 10  DayOfWeek          1000 non-null   int64  
 11  Time_Slot          1000 non-null   str    
 12  Vehicle            1000 non-null   str    
 13  Road_Type          1000 non-null   str    
 14  Promo              1000 non-null   int64  
 15  Tip_Level          1000 non-null   int64  
 16  Delivery_Time_min  1000 non-null   f

In [6]:
# 데이터 스케일링
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# 타겟과 ID 컬럼 설정
target = "Delivery_Time_min"
id_col = "Order_ID"

X = train.drop(columns=[target, id_col]) # 타겟과 ID 컬럼 제거
y = train[target]

In [7]:
# 범주형과 수치형 컬럼 구분
cat_cols = X.select_dtypes(include=["object"]).columns
num_cols = X.select_dtypes(include=["int64", "float64"]).columns

print("범주형 컬럼:", cat_cols)
print("수치형 컬럼:", num_cols)

범주형 컬럼: Index(['Weather', 'Time_Slot', 'Vehicle', 'Road_Type'], dtype='str')
수치형 컬럼: Index(['Distance_km', 'Prep_Time_min', 'Riders_Available', 'Order_Queue',
       'Rain_mm', 'Temp_C', 'Wind_mps', 'Rush_Hour', 'DayOfWeek', 'Promo',
       'Tip_Level'],
      dtype='str')


C:\Users\user\AppData\Local\Temp\ipykernel_26476\701535330.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=["object"]).columns
